# RoPE Frequency-Band Ablation — Kaggle runner

Settings: **Accelerator = GPU T4 x2** (not P100), **Internet = On** (needs phone-verified account; restart the session after turning it on). Add-ons → Secrets: `GITHUB_TOKEN`.

**Run All is safe:** every step goes through `sh()`, which raises on a non-zero exit, and every gate (G0, G2, G3, G4) exits non-zero on failure — so Run All stops at the first failed gate instead of burning GPU quota on a sweep that means nothing.

For the full run use *Save Version → Save & Run All (Commit)* (background, ≤12 h, output kept).
**Resume:** attach the previous version's output as input; cell 2 restores `results/` and finished cells are skipped.

In [ ]:
# 1. Setup — edit these
REPO_URL = "https://github.com/leeduy0403/rope-freq-ablation.git"
PRIVATE_REPO = True       # True -> Kaggle secret GITHUB_TOKEN
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CALIB_FACTORS = (2, 8, 32)
CALIB_TASK, CALIB_LENGTH = "vt", 2048

import os, subprocess, sys, glob, json, shutil

def sh(cmd):
    """Run a shell command, stream its output, raise if it fails (stops Run All)."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
REPO = "/kaggle/working/rope-freq-ablation"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q", url, "main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", url, REPO], check=True)
os.chdir(REPO)
sh("git log --oneline -1 && pip install -q pyyaml")
M = f"--model-id {MODEL_ID}"


In [ ]:
# 2. Restore results from a previous version (if attached as input)
for src in glob.glob("/kaggle/input/*/rope-freq-ablation/results") + glob.glob("/kaggle/input/*/results"):
    print("restoring from", src)
    shutil.copytree(src, "results", dirs_exist_ok=True)


In [ ]:
# Phase 0 — environment (gate G0) + unit tests
sh("python -m src.env_report")
sh("python -m pytest -q tests")


In [ ]:
# Phase 1 — model candidates (config only) + rotary module inspection
sh("python -m src.select_model Qwen/Qwen2.5-0.5B-Instruct Qwen/Qwen2.5-1.5B-Instruct Qwen/Qwen2.5-3B-Instruct")
sh(f"python -m src.rope.patch --inspect {M}")


In [ ]:
# Smoke test — 5 examples, shortest length. Gates not enforced (n too small to mean anything).
sh(f"python -m src.eval.runner --stage baseline {M} --tasks s_niah vt fwe --lengths 1024 --n 5 --results-dir results/_smoke --no-gate")
sh(f"python -m src.eval.runner --stage sweep {M} --tasks s_niah --lengths 1024 --n 5 --bands 0 7 --results-dir results/_smoke --no-gate")


In [ ]:
# FWE difficulty probe (spec G2 remedy 1: stronger Zeta skew).
# At alpha=2.0 the 1.5B model gets top-1/top-2 right but confuses rank 3 vs 4 (smoke: 0/5).
# Rule: smallest alpha with fwe@1024 >= FWE_MIN_ACC (margin over the 0.3 gate); recorded in results/<model>/fwe_alpha.json.
FWE_ALPHAS = (2.0, 2.5, 3.0)
FWE_MIN_ACC = 0.4
slug = MODEL_ID.replace("/", "__")
probe = {}
for a in FWE_ALPHAS:
    print(f"\n===== fwe alpha={a} =====")
    sh(f"python -m src.eval.runner --stage baseline {M} --tasks fwe --lengths 1024 4096 --n 20 "
       f"--set task_params.fwe.alpha={a} --results-dir results/_fwe_a{a} --no-gate")
    probe[a] = {L: json.load(open(f"results/_fwe_a{a}/{slug}/baseline/fwe_{L}.json"))["accuracy"] for L in (1024, 4096)}
ok = [a for a in FWE_ALPHAS if probe[a][1024] >= FWE_MIN_ACC]
os.makedirs(f"results/{slug}", exist_ok=True)
json.dump({"probe": probe, "chosen": ok[0] if ok else None, "min_acc": FWE_MIN_ACC},
          open(f"results/{slug}/fwe_alpha.json", "w"), indent=2)
for a in FWE_ALPHAS:
    print(f"alpha={a}: acc@1024={probe[a][1024]:.2f}  acc@4096={probe[a][4096]:.2f}")
if not ok:
    raise RuntimeError("fwe too hard at every alpha — next remedy: add an in-context example (spec G2 remedy 2)")
FWE_ALPHA = ok[0]
S = f"--set task_params.fwe.alpha={FWE_ALPHA}"
print("chosen fwe alpha =", FWE_ALPHA)


In [ ]:
# Phase 2 — baseline. Gate G2: shortest length s_niah>0.6, vt>0.3, fwe>0.3 (stops here if it fails)
sh(f"python -m src.eval.runner --stage baseline {M} {S}")


In [ ]:
# Phase 3a — patch correctness. Gate G3
sh(f"python -m src.check_patch {M} {S} --tasks s_niah vt --n 20")


In [ ]:
# Phase 3b — calibrate compress_factor on one task/length.
# Rule: pick the factor whose per-band accuracies (bands 1..n-1) are most spread out (largest std).
# Band 0 is excluded: compressing it breaks the model at every factor and would dominate the std.
# too weak -> all bands ~ baseline (flat), too strong -> all bands collapse (flat again).
for f in CALIB_FACTORS:
    sh(f"python -m src.eval.runner --stage sweep {M} {S} --tasks {CALIB_TASK} --lengths {CALIB_LENGTH} --factor {f}")

import numpy as np
slug = MODEL_ID.replace("/", "__")
spread = {}
for f in CALIB_FACTORS:
    accs = [json.load(open(p))["accuracy"] for p in
            glob.glob(f"results/{slug}/sweep/compress_f{f:g}/{CALIB_TASK}_{CALIB_LENGTH}_band[1-9]*.json")]
    spread[f] = float(np.std(accs))
    print(f"factor {f:>4}: per-band acc std = {spread[f]:.3f}  (min {min(accs):.2f}, max {max(accs):.2f})")
FACTOR = max(CALIB_FACTORS, key=lambda f: (spread[f], f == 8))
json.dump({"spread": spread, "chosen": FACTOR, "task": CALIB_TASK, "length": CALIB_LENGTH},
          open(f"results/{slug}/calibration.json", "w"), indent=2)
if spread[FACTOR] < 0.02:
    raise RuntimeError("all factors give a flat profile — methodology problem, stop and inspect (spec §5)")
print("chosen compress_factor =", FACTOR)


In [ ]:
# Phase 3c — full sweep with the chosen factor. Gate G4: band=none reproduces baseline
sh(f"python -m src.eval.runner --stage sweep {M} {S} --factor {FACTOR}")


In [ ]:
# Phase 4/5 — compression profiles, heatmap, overlay, correlations
TAG = f"compress_f{FACTOR:g}"
sh(f"python -m src.analyze.heatmap {M} --tag {TAG}")
sh(f"python -m src.analyze.overlay {M} --tag {TAG} --scale 4")
from IPython.display import Image, display
for p in sorted(glob.glob("figures/*.png")):
    print(p); display(Image(p))


In [ ]:
# Package results for download
sh("cd /kaggle/working && zip -qr results.zip rope-freq-ablation/results rope-freq-ablation/figures && ls -la results.zip")
